# The four labels as a time series: when to send a frame to the VLM

Every CSV row is one frame pair, so each video is already a **label time series**: sort its
rows by frame and you get TRANSITION / PURSUIT / REFIXATION / STABLE, one per second. No
download, no DINOv2, no training: everything here reads files already on Drive.

**The idea under test:** don't send every changing frame. While the scene is moving, wait;
send **one frame when the view settles**.

```
labels:   S  S  T  T  T  S  S  R  S
send A:         ^  ^  ^        ^         every non-STABLE frame (the design PDF's gate)
send C:                  ^               first settled frame after the moving stretch
```

| state | labels | meaning |
|---|---|---|
| **moving** | TRANSITION, PURSUIT | frame similarity low: the head/scene is changing |
| **settled** | REFIXATION, STABLE | frame similarity high: the scene holds still |

| policy | sends frame t when |
|---|---|
| **A** all but STABLE | label is not STABLE (current gate) |
| **B** TRANSITION + REFIX | label is TRANSITION or REFIXATION (discard PURSUIT + STABLE) |
| **C** settle | first settled label after a moving stretch |
| **D** settle + refix | C, plus every REFIXATION inside a settled stretch (new gaze target) |
| **E** D + timeout | D, plus one frame every 5 s during a long moving stretch (walking) |
| **F** E + hysteresis | E with P(moving) thresholds 0.6 in / 0.4 out (models with probabilities) |

## Questions, in order

1. **How long are the runs?** If TRANSITION runs are mostly 1 frame, C saves little over A.
   If they are often 3+ frames, it saves a lot.
2. **What ends a moving run:** STABLE, REFIXATION, or the end of the video?
3. **Replay the policies on the ground-truth labels:** the best each policy could do.
4. **Replay them on each model's predictions** for the 12 test videos.
5. **Look at one video as a timeline.**

## Metrics (always judged against the DINOv2 labels)

| metric | meaning |
|---|---|
| sends / min | compute: frames sent per minute of video |
| settles caught % | true moving → settled boundaries that got a sent frame within ±1 s |
| send offset (s) | when caught: −1 = the frame before the settle, 0 = at it, +1 = one late |
| refix caught % | true REFIXATIONs inside a settled stretch that got a send at or 1 s after |
| mid-motion sends % | sends in the middle of a moving stretch (likely blurred, not attended) |
| **stale frames %** | frames whose CLS cosine to the **last sent** frame is below τ_frame: the VLM's latest view no longer matches what the wearer sees. Policy-neutral; needs `feat_cache.npz` |

## 1. Clone, mount, paths

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('eyehead/policy.py'), f'eyehead/policy.py is not on branch {BRANCH!r}'

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/GazeVLM'
FOLDS      = os.path.join(DRIVE_DIR, 'folds')
THRESH     = os.path.join(DRIVE_DIR, 'thresholds.json')
FEAT_CACHE = os.path.join(DRIVE_DIR, 'feat_cache.npz')        # CLS per frame, from gatetrain
RUNS       = os.path.join(DRIVE_DIR, 'runs')
BASELINE   = os.path.join(DRIVE_DIR, 'release', 'gate_v1', 'test_preds.csv')
OUT        = os.path.join(RUNS, 'policy')
os.makedirs(OUT, exist_ok=True)

for p in (os.path.join(FOLDS, 'train_val_fold1.csv'), os.path.join(FOLDS, 'test.csv'), THRESH):
    assert os.path.exists(p), f'missing {p}'
print('feature cache:', 'found' if os.path.exists(FEAT_CACHE)
      else 'NOT found -- the stale-frames metric will be skipped')
print('old gate preds:', 'found' if os.path.exists(BASELINE) else 'not found')

## 2. Run everything

Reads the labels of all 70 videos (fold file + test file), splits each video into clips of
chained frame pairs, and replays every policy on the ground truth and on each model's test
predictions (`test_preds.csv` from the eyehead notebook). Results are also written to
`MyDrive/GazeVLM/runs/policy/`.

In [ ]:
from eyehead import policy as pol
from eyehead.features import NAMES

PREDS = {'eye only': 'eyehead_eye_only', 'IMU only': 'eyehead_head_only',
         'eye + IMU': 'eyehead_eye_head', 'full': 'eyehead'}
PREDS = {k: os.path.join(RUNS, v, 'test_preds.csv') for k, v in PREDS.items()}
TIMEOUT, HYST = 5, (0.6, 0.4)

o = pol.run_all(FOLDS, THRESH, FEAT_CACHE, PREDS, BASELINE, TIMEOUT, HYST)
res = o['results']

for fname, key in (('runs.csv', 'runs'), ('run_lengths_all.csv', 'lengths_all'),
                   ('run_lengths_test.csv', 'lengths_test'), ('policy_results.csv', 'results')):
    o[key].to_csv(os.path.join(OUT, fname), index=False)
o['moving_next_all'].to_csv(os.path.join(OUT, 'moving_run_next.csv'))
o['transitions_all'].to_csv(os.path.join(OUT, 'transitions.csv'))
print('wrote', OUT)

## 3. How long do the labels last?

`pct_runs_ge3` is the number that decides question 1: the share of runs lasting 3 s or
more. `pct_frames_in_runs_ge3` is the share of that label's frames that sit in such runs,
which is roughly what C can save over A.

In [ ]:
print('RUN LENGTHS, all videos (1 frame = 1 s)')
display(o['lengths_all'].round(2))
print('RUN LENGTHS, test videos')
display(o['lengths_test'].round(2))

rt = o['runs']
COL4 = {'TRANSITION': 'tab:blue', 'PURSUIT': 'tab:orange', 'REFIXATION': 'tab:red', 'STABLE': 'tab:green'}
bins = np.arange(1, 17) - 0.5

fig, ax = plt.subplots(1, 3, figsize=(20, 4.6))
for c in NAMES.values():
    L = rt[(rt.kind == 'class') & (rt.run == c)].length.clip(upper=15)
    ax[0].hist(L, bins=bins, histtype='step', lw=1.8, color=COL4[c], density=True, label=c)
ax[0].set_xlabel('run length in s (15 = 15+)'); ax[0].set_ylabel('share of runs')
ax[0].set_title('how long each label lasts'); ax[0].legend(fontsize=8)

for st, col in (('moving', 'tab:purple'), ('settled', 'tab:gray')):
    L = rt[(rt.kind == 'state') & (rt.run == st)].length.clip(upper=15)
    ax[1].hist(L, bins=bins, histtype='step', lw=2, color=col, density=True, label=st)
ax[1].set_xlabel('run length in s (15 = 15+)')
ax[1].set_title('moving {TRANSITION, PURSUIT} vs settled {REFIXATION, STABLE}'); ax[1].legend()

mv = rt[(rt.kind == 'state') & (rt.run == 'moving')]
ct = pd.crosstab(mv['mix'], mv['next'], normalize='index') * 100
order = [c for c in ('STABLE', 'REFIXATION', 'END') if c in ct.columns]
ct[order].plot(kind='bar', stacked=True, ax=ax[2], rot=0,
               color=[{'STABLE': 'tab:green', 'REFIXATION': 'tab:red', 'END': 'tab:gray'}[c] for c in order])
ax[2].set_ylabel('% of moving runs'); ax[2].set_title('what ends a moving run')
plt.tight_layout(); plt.show()

print('what ends a moving run (counts, all videos; rows = what the run was made of):')
display(o['moving_next_all'])
tm = o['transitions_all']
print('label -> next label, % of each row (all videos):')
display((100 * tm.div(tm.sum(1), axis=0)).round(1))

## 4. The policies on the ground-truth labels

This is the **ceiling** for each policy: what it would do with perfect labels. Compare C/D/E
against A on sends per minute, and check that stale frames % does not rise much.

In [ ]:
COLS = ['sends / min', 'send %', 'settles caught %', 'send offset (s)', 'refix caught %',
        'mid-motion sends %', 'stale frames %', 'mean 1 - cos to last sent']
COLS = [c for c in COLS if c in res.columns]
gt = res[res.source.str.startswith('ground truth')]
display(gt.set_index(['source', 'policy'])[COLS].round(2))

metrics = ['sends / min'] + [m for m in ('stale frames %', 'mid-motion sends %') if m in res.columns]
fig, ax = plt.subplots(1, len(metrics), figsize=(6.5 * len(metrics), 4.4))
for a_, m in zip(np.atleast_1d(ax), metrics):
    gt.pivot(index='policy', columns='source', values=m).plot(kind='barh', ax=a_)
    a_.invert_yaxis(); a_.set_title(m); a_.set_ylabel('')
plt.tight_layout(); plt.show()

## 5. The policies on each model's predictions (12 test videos)

Same policies, now driven by predicted labels, still judged against the true ones. The
scatter puts compute (x) against staleness (y): **lower-left is better**. Black markers are
the ground-truth ceiling.

In [ ]:
mods = res[~res.source.str.startswith('ground truth')]
display(mods.set_index(['source', 'policy'])[COLS].round(2))

y_metric = 'stale frames %' if 'stale frames %' in res.columns else 'settles caught %'
SRC_COL = {'ground truth, test': 'k', 'old gate': 'tab:gray', 'eye only': 'tab:orange',
           'IMU only': 'tab:green', 'eye + IMU': 'tab:purple', 'full': 'tab:blue'}
pols = sorted(res.policy.unique())
MARK = dict(zip(pols, 'osD^vP*X'))

fig, ax = plt.subplots(figsize=(10, 6.5))
for _, r in res[res.source != 'ground truth, all videos'].iterrows():
    ax.scatter(r['sends / min'], r[y_metric], marker=MARK[r.policy], s=80,
               color=SRC_COL.get(r.source, 'tab:red'), edgecolor='k', linewidth=.4)
h1 = [Line2D([], [], marker='o', ls='', color=c, label=s) for s, c in SRC_COL.items()
      if s in set(res.source)]
h2 = [Line2D([], [], marker=MARK[p], ls='', color='k', label=p) for p in pols]
ax.add_artist(ax.legend(handles=h1, loc='upper right', fontsize=8, title='source'))
ax.legend(handles=h2, loc='lower right', fontsize=8, title='policy')
ax.set_xlabel('sends per minute  (compute)'); ax.set_ylabel(y_metric)
ax.set_title('compute vs staleness on the test videos: lower-left is better'); ax.grid(alpha=.3)
plt.show()

## 6. One test video as a timeline

Top row: the true label of every second. Below: each vertical tick is a frame sent to the
VLM. Change `CLIP` to look at another test video (it defaults to the longest).

In [ ]:
clips = o['clips']
test_ids = [i for i, c in enumerate(clips) if c['is_test']]
CLIP = max(test_ids, key=lambda i: len(clips[i]['labels']))

c = clips[CLIP]
lab, n = c['labels'], len(c['labels'])
SHOW = [('ground truth, test', 'A: all but STABLE'),
        ('ground truth, test', 'C: settle'),
        ('ground truth, test', f'E: D + {TIMEOUT}s timeout'),
        ('full', 'A: all but STABLE'),
        ('full', f'E: D + {TIMEOUT}s timeout'),
        ('full', f'F: E + hysteresis {HYST[0]:g}/{HYST[1]:g}')]
SHOW = [s for s in SHOW if (s[0], s[1], CLIP) in o['sent']]

fig, ax = plt.subplots(len(SHOW) + 1, 1, figsize=(19, 1.0 * len(SHOW) + 1.8), sharex=True)
ax[0].bar(np.arange(1, n + 1), 1, width=1.0, color=[COL4[NAMES[v]] for v in lab])
ax[0].set_yticks([])
ax[0].set_ylabel('true label', rotation=0, ha='right', va='center')
ax[0].legend([plt.Rectangle((0, 0), 1, 1, color=COL4[k]) for k in NAMES.values()],
             list(NAMES.values()), ncol=4, fontsize=8, loc='lower right', bbox_to_anchor=(1, 1.02))
ax[0].set_title('%s   (%d s)' % (c['sequence'], n + 1), loc='left')
for a_, (src, p) in zip(ax[1:], SHOW):
    m = o['sent'][(src, p, CLIP)]
    a_.vlines(np.flatnonzero(m), 0, 1, color='k', lw=1.1)
    a_.set_yticks([])
    a_.set_ylabel('%s\n%s\n%d sends' % (src, p, m.sum()), rotation=0, ha='right', va='center', fontsize=8)
ax[-1].set_xlabel('frame (s)')
plt.tight_layout(); plt.show()

---

## Reading it

| look at | what it tells you |
|---|---|
| `pct_runs_ge3` for TRANSITION and PURSUIT | whether the settle idea can save much at all |
| what ends a moving run | how often the view settles into STABLE vs REFIXATION; END means a long motion the timeout must cover |
| sends / min, A vs C/D/E on ground truth | the compute the settle policy saves, with perfect labels |
| stale frames %, A vs C/D/E | what that saving costs: how often the VLM's latest view is out of date |
| mid-motion sends % | A sends many frames mid-motion; C/D should send almost none |
| model rows vs ground-truth rows | how much of the ceiling each model reaches |
| F vs E | whether hysteresis removes spurious sends caused by single-frame mispredictions |

## Caveats

- 1 fps: every decision lands on a 1 s grid, and "the view settled" is only known one frame
  after it happens.
- Stale frames % uses the whole-frame CLS only. A gaze-region change inside an unchanged
  scene (REFIXATION) does not make a frame stale by this measure; refix caught % covers that.
- The labels are DINOv2 proxies. Whether a VLM answers better with these frames is still
  untested.